# Exploratory Data Analysis — 30-Day Hospital Readmission Risk

## Purpose

This notebook provides an interactive walkthrough of the synthetic patient data used to train the readmission prediction model. It is designed to be run on an **Azure ML Compute Instance** in the **dev** workspace.

### What we'll cover:
1. **Connect to the workspace** and retrieve data directly from the registered data asset
2. **Inspect the dataset** — shape, types, missing values
3. **Explore the target variable** — class balance and readmission rate
4. **Visualise numeric features** — distributions and outliers
5. **Visualise categorical features** — frequency counts
6. **Analyse correlations** — which features relate most to readmission
7. **Readmission rate by category** — stratified views for clinical insight

### Prerequisites
- Running on an Azure ML Compute Instance (or local with `azure-ai-ml` installed)
- The `readmission-raw-data` data asset must be registered in the dev workspace
  (run `python scripts/generate_and_upload_data.py -g rg-readmit-dev -w <workspace-name>` first)

In [ ]:
# ============================================================================
# Cell 1: Import Libraries
# ============================================================================
# Core data manipulation and visualisation libraries.
# We also import the Azure ML SDK v2 to connect to the workspace and
# retrieve the registered data asset programmatically.

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# Set visual defaults for clean, consistent plots
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 25)

In [ ]:
# ============================================================================
# Cell 2: Connect to Azure ML Workspace & Load Data Asset
# ============================================================================
# Instead of reading a local CSV, we connect to the dev workspace and pull
# the registered data asset. This ensures we're exploring the exact same data
# that the training pipeline consumes.
#
# DefaultAzureCredential works automatically on Compute Instances (managed identity)
# and locally if you've run `az login`.

# Connect to the dev workspace
# On a Compute Instance, subscription/RG/workspace are auto-detected from config.
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print(f"Connected to workspace: {ml_client.workspace_name}")

# Retrieve the latest version of the registered data asset
data_asset = ml_client.data.get(name="readmission-raw-data", label="latest")
print(f"Data asset: {data_asset.name} (version {data_asset.version})")
print(f"Path: {data_asset.path}")

# Load into a DataFrame — the path points to a folder containing CSV file(s)
df = pd.read_csv(data_asset.path)
print(f"\nDataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 1. Target Distribution — Class Balance

The target variable `readmitted_30d` is binary (0 = not readmitted, 1 = readmitted within 30 days).

**Why this matters:** Class imbalance directly affects model training strategy. If readmission is rare (~20%), we may need:
- Stratified train/test splits
- Class weights or oversampling
- Metrics beyond accuracy (AUC-ROC, F1, Precision-Recall)

In [ ]:
# ============================================================================
# Cell 3: Target Variable Distribution
# ============================================================================
# Visualise the balance between readmitted and not-readmitted patients.

TARGET_COL = "readmitted_30d"

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Count plot
target_counts = df[TARGET_COL].value_counts()
target_counts.plot(kind="bar", ax=axes[0], color=["steelblue", "coral"])
axes[0].set_title("Count of Readmission Classes")
axes[0].set_xlabel("Readmitted within 30 days")
axes[0].set_ylabel("Count")
axes[0].set_xticklabels(["No (0)", "Yes (1)"], rotation=0)

# Percentage annotation
for i, (val, count) in enumerate(target_counts.items()):
    pct = count / len(df) * 100
    axes[0].text(i, count + 50, f"{pct:.1f}%", ha="center", fontweight="bold")

# Pie chart for quick visual
target_counts.plot(kind="pie", ax=axes[1], labels=["Not Readmitted", "Readmitted"],
                   autopct="%1.1f%%", colors=["steelblue", "coral"], startangle=90)
axes[1].set_ylabel("")
axes[1].set_title("Readmission Rate")

plt.tight_layout()
plt.show()

print(f"\nReadmission rate: {df[TARGET_COL].mean():.1%}")
print(f"Class ratio (negative:positive): {target_counts[0] / target_counts[1]:.1f}:1")

## 2. Numeric Feature Distributions

We have **12 numeric features** covering demographics, clinical measurements, and utilisation metrics.

**What to look for:**
- **Skewness** — heavily skewed features may benefit from log/power transforms
- **Outliers** — extreme values that could dominate tree splits or distance-based models
- **Modality** — bimodal distributions may suggest hidden sub-populations
- **Scale differences** — features on very different scales may need normalisation for some models

In [ ]:
# ============================================================================
# Cell 4: Numeric Feature Histograms
# ============================================================================
# Plot distribution of each numeric feature, coloured by readmission status
# to see if any features have obvious separability.

NUMERIC_COLS = [
    "age", "num_prior_admissions", "length_of_stay_days", "num_diagnoses",
    "num_procedures", "num_medications", "num_lab_results",
    "days_since_last_admission", "bmi", "heart_rate_avg", "systolic_bp_avg", "hba1c",
]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    ax = axes[i]
    for label, colour in [(0, "steelblue"), (1, "coral")]:
        subset = df[df[TARGET_COL] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=f"{'No' if label == 0 else 'Yes'}", color=colour)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle("Numeric Feature Distributions by Readmission Status", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Categorical Feature Distributions

We have **5 categorical features** covering demographics and administrative details.

**What to look for:**
- **Cardinality** — too many unique values can cause sparse one-hot encodings
- **Dominant classes** — if one category accounts for 90%+ of observations, it may not be informative
- **Potential data quality issues** — unexpected categories, misspellings, etc.

In [ ]:
# ============================================================================
# Cell 5: Categorical Feature Bar Charts
# ============================================================================
# Show value counts for each categorical feature.

CATEGORICAL_COLS = ["gender", "admission_type", "discharge_disposition",
                    "primary_diagnosis_group", "payer_code"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(CATEGORICAL_COLS):
    ax = axes[i]
    df[col].value_counts().plot(kind="bar", ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(f"{col} (n_unique={df[col].nunique()})", fontsize=10)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)

# Hide the unused subplot
axes[-1].set_visible(False)

plt.suptitle("Categorical Feature Distributions", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis

### Numeric Features vs. Target

We compute point-biserial correlations between each numeric feature and the binary target. This tells us the **linear** relationship strength — note that tree-based models can capture non-linear effects that won't show up here.

**Interpretation guide:**
- |r| > 0.3 → moderate linear relationship
- |r| > 0.5 → strong linear relationship
- Sign indicates direction (positive = higher value → more readmissions)

In [ ]:
# ============================================================================
# Cell 6: Correlation with Target
# ============================================================================
# Point-biserial correlation between numeric features and binary target.
# Sorted by absolute value to highlight the most predictive features.

correlations = df[NUMERIC_COLS + [TARGET_COL]].corr()[TARGET_COL].drop(TARGET_COL)
correlations_sorted = correlations.reindex(correlations.abs().sort_values(ascending=False).index)

fig, ax = plt.subplots(figsize=(10, 5))
colours = ["coral" if x > 0 else "steelblue" for x in correlations_sorted]
correlations_sorted.plot(kind="barh", ax=ax, color=colours, edgecolor="white")
ax.set_xlabel("Correlation with readmitted_30d")
ax.set_title("Numeric Features — Correlation with Readmission (sorted by |r|)")
ax.axvline(x=0, color="black", linewidth=0.8)

# Annotate values
for i, (idx, val) in enumerate(correlations_sorted.items()):
    ax.text(val + 0.01 if val >= 0 else val - 0.01, i, f"{val:.3f}",
            va="center", ha="left" if val >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.show()

print("\nTop 5 features by absolute correlation:")
print(correlations_sorted.head().to_string())

## 5. Readmission Rate by Category

Breaking the readmission rate down by each categorical feature reveals **which patient groups** are most at risk. This is clinically actionable insight — it helps identify where interventions could reduce readmissions.

**Note:** These are observational associations, not causal claims.

In [ ]:
# ============================================================================
# Cell 7: Readmission Rate by Category
# ============================================================================
# For each categorical feature, compute and plot the readmission rate per group.

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

overall_rate = df[TARGET_COL].mean()

for i, col in enumerate(CATEGORICAL_COLS):
    ax = axes[i]
    rates = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="coral", edgecolor="white")
    ax.axhline(y=overall_rate, color="black", linestyle="--", linewidth=1, label=f"Overall ({overall_rate:.1%})")
    ax.set_title(f"Readmission Rate by {col}", fontsize=10)
    ax.set_ylabel("Readmission Rate")
    ax.set_ylim(0, min(1.0, rates.max() * 1.5))
    ax.legend(fontsize=8)
    ax.tick_params(axis="x", rotation=45)

axes[-1].set_visible(False)

plt.suptitle("Readmission Rate Stratified by Categorical Features", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 6. Data Quality — Summary Statistics & Missing Values

A quick sanity check before modelling. We look for:
- **Missing values** — do any features have nulls that need imputation?
- **Descriptive stats** — are min/max values clinically plausible?
- **Zero-variance features** — columns with a single unique value provide no signal

In [ ]:
# ============================================================================
# Cell 8: Summary Statistics
# ============================================================================
# Descriptive statistics for numeric features — check for plausible ranges.

print("=" * 60)
print("NUMERIC FEATURE SUMMARY")
print("=" * 60)
display(df[NUMERIC_COLS].describe().T.style.format("{:.2f}"))

print("\n" + "=" * 60)
print("CATEGORICAL FEATURE SUMMARY")
print("=" * 60)
for col in CATEGORICAL_COLS:
    print(f"\n{col}: {df[col].nunique()} unique values")
    print(f"  Most common: {df[col].mode()[0]} ({df[col].value_counts().iloc[0]} occurrences)")
    print(f"  Least common: {df[col].value_counts().index[-1]} ({df[col].value_counts().iloc[-1]} occurrences)")

## 7. Missing Values Check

Check for any null/NaN values across the dataset. The training pipeline's `prep.py` step handles imputation, but understanding the extent of missingness here informs whether our imputation strategy is adequate.

In [ ]:
# ============================================================================
# Cell 9: Missing Values Analysis
# ============================================================================
# Check for any missing values and visualise their distribution.

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)

if missing_df.empty:
    print("✓ No missing values found in the dataset!")
    print(f"  Total cells: {df.shape[0] * df.shape[1]:,}")
else:
    print(f"⚠ {len(missing_df)} columns have missing values:\n")
    display(missing_df)
    
    # Visualise missing pattern
    fig, ax = plt.subplots(figsize=(10, 4))
    missing_df["Missing %"].plot(kind="barh", ax=ax, color="coral")
    ax.set_xlabel("% Missing")
    ax.set_title("Missing Values by Feature")
    plt.tight_layout()
    plt.show()

print(f"\nDataset overview: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")